<a href="https://colab.research.google.com/github/SUZUB/ATU/blob/main/ATU_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Cell 1: Infrastructure & Cloud Sync
# ==============================================================================
import os
from google.colab import drive

# 1. Public Git clone / pull
if not os.path.exists('ATU') and not os.path.exists('/content/ATU'):
    !git clone https://github.com/SUZUB/ATU.git
if os.path.exists('/content/ATU'):
    %cd /content/ATU
elif os.path.exists('ATU'):
    %cd ATU
# Force-sync to the latest GitHub version. /content/ATU is a disposable mirror:
# edit code locally and push; any edits made here are discarded (listed first).
!git fetch -q origin && git status --short && git reset -q --hard origin/main
!git log -1 --format="Repo synced to commit %h (%cd): %s" --date=short

# Install project requirements
!pip install -q -r requirements.txt

# yfinance >= 1.8 requires curl_cffi >= 0.17 (yfinance's "chrome" profile resolves to
# chrome152). Upgrade both together; an older curl_cffi still loaded in this runtime
# is what raises ImpersonateError('Impersonating chrome152 is not supported').
!pip install -q -U "yfinance>=1.8.0" "curl_cffi>=0.17.0"

import sys, subprocess, importlib.metadata as md
stale = [
    pkg for pkg in ("curl_cffi", "yfinance")
    if pkg in sys.modules and getattr(sys.modules[pkg], "__version__", None) != md.version(pkg)
]
if stale:
    print(f"Outdated {stale} already loaded in this runtime -> restarting. Re-run from Cell 1 afterwards.")
    os.kill(os.getpid(), 9)  # Colab reconnects automatically with the upgraded packages

# Smoke test in a fresh process, using the same session logic as the pipeline
# (market_data_processor.get_yfinance_session picks a browser profile curl_cffi supports)
probe = subprocess.run(
    [sys.executable, "-c",
     "import yfinance as yf, curl_cffi, market_data_processor as m; "
     "d = m._yf_download(tickers='AAPL', period='5d', interval='5m', progress=False); "
     "print(f'yfinance {yf.__version__} | curl_cffi {curl_cffi.__version__} | "
     "profile {m._YF_SESSION_STATE.get(\"profile\")} | AAPL 5m rows: {len(d)}')"],
    capture_output=True, text=True,
)
print(probe.stdout.strip() or f"yfinance smoke test FAILED:
{probe.stderr.strip()[-1500:]}")

# 2. Google Drive Mount
drive.mount('/content/drive')

# 3. Persistent Drive Workspace & Subdirectory Creation
DRIVE_WORKSPACE = '/content/drive/MyDrive/ATU_Workspace'
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

DATASET_PATH = os.path.join(DRIVE_WORKSPACE, 'datasets')
MODEL_CHECKPOINTS = os.path.join(DRIVE_WORKSPACE, 'checkpoints')
RESULTS_PATH = os.path.join(DRIVE_WORKSPACE, 'results')

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(MODEL_CHECKPOINTS, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("\nInfrastructure & Cloud Sync complete.")
print(f"Project Directory: {os.getcwd()}")
print(f"Drive Workspace:   {DRIVE_WORKSPACE}")

In [ ]:
# ==============================================================================
# Cell 2: Imports & Setup
# ==============================================================================
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Ensure ATU root directory is in sys.path
atu_root = Path.cwd()
if str(atu_root) not in sys.path:
    sys.path.insert(0, str(atu_root))

# Drop cached project modules so code pulled in Cell 1 is used without a runtime restart
_project_modules = ("market_data_processor", "sentiment_processor", "constrained_trading_env", "train_and_backtest", "research_strategies", "orb_model")
for _name in [m for m in sys.modules if m in _project_modules or m == "src" or m.startswith("src.")]:
    del sys.modules[_name]

# Undo any yf.download monkey-patch left over from an earlier cell (re-running such a
# patch makes it wrap itself -> RecursionError). The pipeline does not need one.
import yfinance as yf
import yfinance.multi
yf.download = yfinance.multi.download

# Shared pipeline (identical logic to `python train_and_backtest.py`)
from src.train import build_live_state_vector, run_experiment
import research_strategies as rs
import orb_model as om

# Device Auto-detection (FinBERT runs on GPU when available; SB3 MLP agents train on CPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Imports & Setup complete.")
print(f"PyTorch Version:  {torch.__version__}")
print(f"Active Device:    {DEVICE.upper()}")
if torch.cuda.is_available():
    print(f"GPU Hardware:     {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# Cell 3: Real Market Data + Real News Sentiment -> final_state_vector
# ==============================================================================
# - Real candles from yfinance. Default: 1-hour bars over ~2 years (~580 training days,
#   ~145 test days). 5-minute bars are limited by Yahoo to the last 60 days (~12 test days,
#   too short to judge a strategy); use INTERVAL = "5m", PERIOD = "60d" for those.
# - Also builds a price-mirrored copy (every return negated) used as extra TRAINING data,
#   so agents cannot profit from a fixed long/short bias.
# - The news step is optional: if it fails, the cell continues with neutral sentiment.
#   Over 2 years the first news fetch takes several minutes; later runs reuse the cache
#   in <workspace>/datasets/news_cache_<TICKER>.csv.
# - FinBERT scores, dated in the exchange timezone, lagged 1 day (Tue news -> Wed candles).
# - Session rules: NSE/BSE (.NS/.BO) 09:15-15:30 IST, square-off 15:15;
#                  US 09:30-16:00 ET, square-off 15:45 (and always at each day's last bar).
TICKER = "AAPL"  # e.g. "RELIANCE.NS"
INTERVAL = "1h"
PERIOD = "730d"
USE_NEWS_SENTIMENT = True  # False = price data only (skips news + FinBERT; sentiment = 0.0)
DRIVE_WORKSPACE = globals().get("DRIVE_WORKSPACE", "/content/drive/MyDrive/ATU_Workspace")

data = build_live_state_vector(
    ticker=TICKER,
    period=PERIOD,
    interval=INTERVAL,
    data_dir=os.path.join(DRIVE_WORKSPACE, "datasets"),
    device=DEVICE,
    use_news=USE_NEWS_SENTIMENT,
)
final_state_vector = data.final_state_vector
print(f"\nEnv Rules: {data.env_kwargs}")
display(final_state_vector.head(3))


In [ ]:
# ==============================================================================
# Cell 4: Multi-Seed Ensemble RL (PPO + A2C) & Backtest
# ==============================================================================
# - One PPO + A2C ensemble per seed, trained on the same 80% (+ its price mirror) and
#   tested on the same 20%. A real edge shows up in most seeds; one good seed is luck.
# - Training budget = TRAIN_PASSES passes over the training data (fixed, small: more
#   passes over the same history = memorization). Fresh models every run (RESUME = False)
#   so each run is a clean, comparable experiment.
# - Rewards are VecNormalize'd for training only; the backtest runs on a raw env.
# - Per-seed reports, daily P&L and plots: <Drive>/results/seed_<n>/
#   Cross-seed summary: <Drive>/results/seed_summary.txt and equity_by_seed.png
SEEDS = [0, 1, 2]
TRAIN_PASSES = 20
RESUME = False

assert "data" in globals(), "Run Cell 3 first."
experiment = run_experiment(
    data,
    workspace=DRIVE_WORKSPACE,
    seeds=SEEDS,
    train_passes=TRAIN_PASSES,
    resume=RESUME,
    show_plot=True,
)
for seed, run in experiment.runs.items():
    print(f"\n----- Seed {seed} -----\n" + run.report)
print("\n" + experiment.report)


In [ ]:
# ==============================================================================
# Cell 5: Research Strategy - 5-Minute Opening Range Breakout (rule-based, no RL)
# ==============================================================================
# Zarattini & Aziz (2023), SSRN 4416622, and Zarattini, Barbon & Aziz (2024), SFI 24-98:
# - First 5-minute candle = opening range. Green -> long at the next bar's open,
#   red -> short, doji -> no trade. Stop = other side of the opening range.
# - Exit at RR x risk (RR = 3 -> 1:3 risk:reward) or at the session close.
# - Each trade risks RISK_PER_TRADE of its share of capital (leverage capped).
# Yahoo keeps 5-minute bars for only 60 days, so every run MERGES new bars into
# <Drive>/datasets/intraday/: the test history grows each time you run this cell.
# Judge it by the consistency lines: both halves positive at YOUR real fee level.
RESEARCH_TICKERS = rs.DEFAULT_UNIVERSE  # or e.g. ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS"]
RR = 3.0                 # 1:3 risk:reward; None = hold to the close
RISK_PER_TRADE = 0.01    # 1% of allocated capital per trade (paper setting)
MAX_LEVERAGE = 4.0       # paper setting; use 1.0 for no leverage
FEE_PER_SIDE = 0.0001    # 0.01% fees + slippage per side; NSE intraday is closer to 0.02-0.03%

intraday = rs.download_universe(
    RESEARCH_TICKERS, period="60d", interval="5m",
    cache_dir=os.path.join(DRIVE_WORKSPACE, "datasets", "intraday"),
)
report_text, orb = rs.research_report(
    intraday, rr=RR, risk_per_trade=RISK_PER_TRADE, max_leverage=MAX_LEVERAGE, fee_rate=FEE_PER_SIDE,
)
research_dir = os.path.join(DRIVE_WORKSPACE, "results", "research")
os.makedirs(research_dir, exist_ok=True)
orb.trades.to_csv(os.path.join(research_dir, "orb_trades.csv"), index=False)
orb.daily.to_csv(os.path.join(research_dir, "orb_daily.csv"))
with open(os.path.join(research_dir, "orb_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)
print(report_text)

plt.figure(figsize=(12, 4))
plt.plot(pd.to_datetime(orb.daily.index), orb.daily["equity"], color="#2563EB", label=orb.name)
plt.axhline(orb.initial_capital, color="gray", linestyle="--", alpha=0.7)
plt.title("5-Minute ORB: equity by session")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# Cell 6: Trained ORB Model - load, update with new data, save, today's signals
# ==============================================================================
# Learns from every (stock, day) which opening-range trades are worth taking:
# features at 09:35 (first 5-min candle, gap, relative volume, previous day, market)
# -> expected net R of a 1:3 bracket LONG and SHORT (gradient-boosted trees).
# Every run: loads the saved model, merges new 5-min bars into Drive (history grows),
# re-checks the features for look-ahead leaks, evaluates walk-forward OUT-OF-SAMPLE
# against plain ORB, retrains on ALL history and saves:
#   <Drive>/models/orb_model.joblib (latest) + a dated copy per run + orb_model.json
# Live signals use the model only if it beat plain ORB in both test halves;
# otherwise they fall back to plain ORB. Run after 09:35 ET for today's signals.
ORB_FEE_PER_SIDE = 0.0001   # your real fees + slippage per side
ORB_MIN_EDGE = 0.10          # trade only when the predicted net gain is above +0.10R

orb_run = om.train_and_save(
    DRIVE_WORKSPACE,
    tickers=om.MODEL_UNIVERSE,
    fee_rate=ORB_FEE_PER_SIDE,
    min_edge=ORB_MIN_EDGE,
)
signals = om.todays_signals(orb_run["model"], orb_run["data"])
print(f"\nSignals for {signals['session'].iloc[0] if len(signals) else 'n/a'} "
      f"(stop = other side of the first 5-min candle, target = 1:3):")
display(signals[signals["signal"] != "SKIP"])

ev = orb_run["evaluation"]
if "model_daily" in ev:
    plt.figure(figsize=(12, 4))
    plt.plot(pd.to_datetime(ev["model_daily"].index), ev["model_daily"]["equity"], label="Model (out-of-sample)")
    plt.plot(pd.to_datetime(ev["baseline_daily"].index), ev["baseline_daily"]["equity"], label="Plain ORB")
    plt.axhline(100_000, color="gray", linestyle="--", alpha=0.7)
    plt.title("Walk-forward equity: trained ORB model vs plain ORB")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
